In [10]:
import warnings
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# insert(0): quando o pacote editable esta' instalado no venv, ele aponta para
# OUTRO checkout; o append deixava o common do venv ganhar deste. Os outros
# notebooks ja' usam insert(0).
sys.path.insert(0, str(Path.cwd().parent / "models" / "src" ))

from common import contract, data, paths, plots
from common.data import read_csv, show_dataset

SEED = contract.SEED
np.random.seed(SEED)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
NL = chr(10)   # evita barra invertida no fonte; usado nos relatorios

DATA_DIR = paths.RAW
OUT_DIR = paths.SPLITS

model_settings = dict()

plots.aplicar_estilo()

print(f"dados em {DATA_DIR}")
print(f"saida em {OUT_DIR}")
print(f"semente: {SEED}")

ARQUIVOS = (
    sorted(DATA_DIR.rglob("ifpee*.csv")) # utilizar apena dados dos inversores e.x
)
print(f"{len(ARQUIVOS)} arquivos" + NL)

# O downsample e as regras vivem em common.data: o MESMO dataframe agregado
# alimenta o split e os limites de sanidade fisica (data.dataset_limits).

dados em /home/scorel/Documents/knob-senses/data/raw
saida em /home/scorel/Documents/knob-senses/data/splits
semente: 42
9 arquivos



In [11]:
partes: list[pd.DataFrame] = []

for p in ARQUIVOS:
    df = read_csv(p, parse=("datetime", "timestamp"))
    df["inversor"] = p.stem.split("_")[0]
    partes.append(df)

    print(f"  {p.parent.name:10} {p.stem:16} {len(df):7} linhas  {len(df.columns):2} colunas")

raw: pd.DataFrame = pd.concat(partes, ignore_index=True)
del partes

  raw        ifpee1_2021        45872 linhas  18 colunas
  raw        ifpee1_2022        43750 linhas  18 colunas
  raw        ifpee1_2023        46380 linhas  18 colunas
  raw        ifpee1_2024        46514 linhas  19 colunas
  raw        ifpee1_2025        44969 linhas  19 colunas
  raw        ifpee2_2024        46514 linhas  19 colunas
  raw        ifpee2_2025        44969 linhas  19 colunas
  raw        ifpee3_2024        46514 linhas  19 colunas
  raw        ifpee3_2025        44969 linhas  19 colunas


In [12]:
raw = raw.drop(columns=[c for c in raw.columns if c in {"condicao", "velocidade_vento", "timestamp"}])

df = raw.copy()

df = df.dropna(subset=['PCA'])

show_dataset(df, colunas=True).head()


pool: 410375 linhas, 16 colunas
colunas: ['PCA', 'angzen', 'datetime', 'doy', 'ghi_cs', 'ghit', 'hora_dec', 'inversor', 'ioeff', 'jacques', 'kc', 'kt', 'pvsyst_cell', 'sandia', 'sandia_cell', 'temperatura_ambiente']

nulos por coluna (so as que tem):
  (nenhum)

colunas com zero:
PCA    9607


,temperatura_ambiente,doy,ghi_cs,hora_dec,kc,angzen,ghit,PCA,jacques,sandia,sandia_cell,pvsyst_cell,datetime,ioeff,kt,inversor
0,29.740,4,971.194471,10.150000,1.058490,25.263877,924.576334,1.28,53.714096,53.072812,60.619963,55.564373,2021-01-04 10:05:00,1411.074617,0.805575,ifpee1
1,30.222,4,976.384198,10.200000,1.059624,24.284699,935.169209,1.70,55.256880,54.528103,62.127197,56.342243,2021-01-04 10:10:00,1411.074617,0.804377,ifpee1
2,30.548,4,984.658595,10.283333,1.080374,23.325885,966.293219,1.72,56.416083,55.663052,63.515055,57.537569,2021-01-04 10:15:00,1411.074617,0.820995,ifpee1
3,30.900,4,992.418037,10.366667,0.846297,22.390090,786.259644,1.70,51.416415,50.859537,57.272172,52.861045,2021-01-04 10:20:00,1411.074617,0.643736,ifpee1
4,29.426,4,999.658520,10.450000,0.742374,21.480359,710.256671,1.00,48.439875,47.886373,53.657849,49.264204,2021-01-04 10:25:00,1411.074617,0.565181,ifpee1


In [13]:
# O filtro de leitura valida vive em common.data: e' o MESMO criterio que
# data.dataset_limits() usa para os limites de sanidade fisica. Assim, mudar o
# filtro aqui muda o limite la' — sem duas copias divergindo em silencio.
df = show_dataset(data.valid_readings(df))


pool: 401820 linhas, 16 colunas

nulos por coluna (so as que tem):
  (nenhum)

colunas com zero:
PCA    1057


## Downsampling dataset

In [14]:
# Regras e frequencia do downsample vivem em common.data: o MESMO dataframe
# agregado alimenta o split e os limites de sanidade fisica.
frequency = '10min'

# PCA (potencia) e ghit (irradiancia) sao intensivas: MEDIA. O alvo e' potencia
# media em kW; somar transformaria em energia e faria o alvo depender do numero
# de amostras validas do bin. As temperaturas tambem sao medias.
df = data.downsample(df, freq=frequency)
show_dataset(df, colunas=True)

# Box-Cox exige alvo positivo, entao os zeros saem aqui. A transformacao em si
# NAO fica no prep: ela e' especifica do MLP e vive em utils/treino_mlp.ipynb,
# com o lambda ajustado so' no treino. Assim o PCA compartilhado (RF, kNN,
# target_limits e a lib C) continua em kW.
df = df[df.PCA >= 0.001]
model_settings['sample_freq'] = frequency
model_settings['target_aggregation'] = 'mean'
model_settings['target_transform'] = False


pool: 202440 linhas, 12 colunas
colunas: ['PCA', 'datetime', 'ghit', 'inversor', 'ioeff', 'jacques', 'kc', 'kt', 'pvsyst_cell', 'sandia', 'sandia_cell', 'temperatura_ambiente']

nulos por coluna (so as que tem):
  (nenhum)

colunas com zero:
PCA    576


## Split temporal

`train` = 2021..2024H1, `calibration` = 2024H2, `test` = 2025. O corte é temporal,
sem embaralhar: o teste é futuro de verdade. A calibração é a janela mais recente
antes do teste; é nela que o viés dos modelos é medido. O teste nunca entra em
treino, escolha de hiperparâmetro nem calibração.

In [15]:
# Corte temporal visivel no notebook: treino ate' 2024-06; calibracao = 2024H2;
# teste = 2025. Sem embaralhar e sem sobreposicao.
CORTE_TREINO = pd.Timestamp("2024-07-01")
ANO_TESTE = int(df.datetime.dt.year.max())
INICIO_TESTE = pd.Timestamp(f"{ANO_TESTE}-01-01")

d2 = df[df.inversor == contract.TRAIN_INVERSOR].copy()

treino = d2[d2.datetime < CORTE_TREINO]
calibracao = d2[(d2.datetime >= CORTE_TREINO) & (d2.datetime < INICIO_TESTE)]
teste = d2[d2.datetime >= INICIO_TESTE]

assert len(treino) and len(calibracao) and len(teste), "split vazio"
assert treino.datetime.max() < calibracao.datetime.min(), "treino encosta na calibracao"
assert calibracao.datetime.max() < teste.datetime.min(), "calibracao encosta no teste"

splits = {"train": treino, "calibration": calibracao, "test": teste}
for nome, frame in splits.items():
    print(f"{nome:12} {len(frame):6} linhas | "
          f"{frame.datetime.min().date()} .. {frame.datetime.max().date()}")

train         78881 linhas | 2021-01-04 .. 2024-06-30
calibration   11335 linhas | 2024-07-01 .. 2024-12-31
test          22619 linhas | 2025-01-01 .. 2025-12-31


In [16]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
escritos = {}

for nome, frame in splits.items():
    caminho = OUT_DIR / f"{nome}.csv"
    frame.to_csv(caminho, index=False)
    escritos[caminho] = len(frame)

# Copias do ultimo ano por inversor (ifpee1/2/3), para comparar as usinas no
# mesmo ano do teste.
teste_dir = OUT_DIR / "test"
teste_dir.mkdir(parents=True, exist_ok=True)
ultimo_ano = df[df.datetime.dt.year == ANO_TESTE]
for inversor, frame in ultimo_ano.groupby("inversor"):
    caminho = teste_dir / f"{inversor}.csv"
    frame.to_csv(caminho, index=False)
    escritos[caminho] = len(frame)

for caminho, n in sorted(escritos.items()):
    print(f"  {caminho.relative_to(paths.ROOT)!s:40} {n:6} linhas")

import json
save_path = paths.ROOT / "model_settings.json"
with open(save_path, "w") as settings:
    json.dump(model_settings, settings)
print(f"\nSaved at {save_path}")

  data/splits/calibration.csv               11335 linhas
  data/splits/test/ifpee1.csv               22619 linhas
  data/splits/test/ifpee2.csv               22618 linhas
  data/splits/test/ifpee3.csv               22618 linhas
  data/splits/test.csv                      22619 linhas
  data/splits/train.csv                     78881 linhas

Saved at /home/scorel/Documents/knob-senses/model_settings.json
